In [ ]:
# !pip install psycopg2-binary -q

In [ ]:
import pandas as pd 
import polars as pl 
from sqlalchemy import create_engine, text
from openhexa.sdk import workspace, current_run 
from datetime import datetime

from sqlalchemy.engine import Engine

import time
import gc

In [ ]:
# helper functions

def check_partition_exist(engine, table: str, year: int) -> bool:
    with engine.begin() as conn:
        exists = conn.execute(text(f'''SELECT 1 FROM pg_class c 
                JOIN pg_inherits i ON c.oid = i.inhrelid 
                JOIN pg_class p ON p.oid = i.inhparent
                WHERE p.relname = '{table}' AND c.relname = '{table}_{year}';''')).scalar()
    return True if exists else False


# log msg to OH
def log_msg(msg, level: str = "info") -> None:
    if "current_run" in globals() and current_run:
        if level == "info":
            current_run.log_info(msg)
        if level == "warning":
            current_run.log_warning(msg)
        if level == "error":
            current_run.log_error(msg)
    print(f"{level} : {msg}")


## Aggregated DSE Data with Health Zone Geometries

We created a **database view** to provide a **pre-aggregated and shape joined dataset** for the DSE surveillance agg superset table update.  
We use this table to refresh the year data (year partitions) in the master table **DSE_Surv_Epi_Complete_agg_ss** used in SuperSet.

- **Aggregates** total cases (`Cas`) and deaths (`Deces`) by:
  - Year  
  - Province  
  - Health Zone (`ZONE`)  
  - Disease (`MALADIE`)  
  - Epidemiological week (`NUMSEM`)  
- **Joins** aggregated data to the corresponding health zone polygons (`first_polygon`) from `cod_iaso_zone_de_sante_ss`.
- **Performance**:
  - Aggregation happens **before** the spatial join
  - Uses PostgreSQL **partition pruning** on the year column

---

### SQL View Definition

```sql
CREATE OR REPLACE VIEW public.view_dse_epi_agg_with_shapes AS
WITH dse_agg AS (
    SELECT
        dse."year",
        dse."PROVINCE",
        dse."ZONE",
        dse."MALADIE",
        dse."NUMSEM",
        dse."zone_id_DHIS2",
        MAX(dse."ALERTE") AS "ALERTE",
        SUM(dse."Cas")   AS "Cas",
        SUM(dse."Deces") AS "Deces"
    FROM public."DSE_Surv_Epi_Complete_agg" dse
    GROUP BY
        dse."year",
        dse."PROVINCE",
        dse."ZONE",
        dse."MALADIE",
        dse."NUMSEM",
        dse."zone_id_DHIS2"
)
SELECT
    a."year",
    a."PROVINCE",
    a."ZONE",
    a."MALADIE",
    a."NUMSEM",
    a."Cas",
    a."Deces",
    a."ALERTE",
    s.first_polygon
FROM dse_agg a
LEFT JOIN public."cod_iaso_zone_de_sante_ss" s
    ON a."zone_id_DHIS2" = s."zone_de_sante_id";


**Usage:**  

```sql
SELECT *  
FROM public.view_dse_epi_agg_with_shapes  
WHERE "year" = 2025;  

In [ ]:
# target table name
source_table = "view_dse_epi_agg_with_shapes"
target_table = "DSE_Surv_Epi_Complete_agg_ss" 

# Database Connection 
engine = create_engine(workspace.database_url)
print(f"Connected to: {engine.url.host}")

**SET up**

In [ ]:
# Create view to get the data from
log_msg("Creating view..")
with engine.begin() as conn:
    conn.execute(text(f'''
    CREATE OR REPLACE VIEW public.view_dse_epi_agg_with_shapes AS
    WITH dse_agg AS (
        SELECT
            dse."year",
            dse."PROVINCE",
            dse."ZONE",
            dse."MALADIE",
            dse."NUMSEM",
            dse."zone_id_DHIS2",
            MAX(dse."ALERTE") AS "ALERTE",
            SUM(dse."Cas")   AS "Cas",
            SUM(dse."Deces") AS "Deces"
        FROM public."DSE_Surv_Epi_Complete_agg" dse
        GROUP BY
            dse."year",
            dse."PROVINCE",
            dse."ZONE",
            dse."MALADIE",
            dse."NUMSEM",
            dse."zone_id_DHIS2"
    )
    SELECT
        a."year",
        a."PROVINCE",
        a."ZONE",
        a."MALADIE",
        a."NUMSEM",
        a."Cas",
        a."Deces",
        a."ALERTE",
        s.first_polygon
    FROM dse_agg a
    LEFT JOIN public."cod_iaso_zone_de_sante_ss" s
        ON a."zone_id_DHIS2" = s."zone_de_sante_id"
    '''))
print(f"Master table '{target_table}' ensured.")

### Start processing

In [ ]:
# Load available years
years_available = pd.read_sql(text(f'SELECT distinct "year" FROM public."{source_table}"'), engine).year.tolist()
years_available = [int(y) for y in sorted(years_available)] 
print(f"Years available in source: {years_available}")

**Select years to process (last year)**  

The years to be refreshed are selected and stored in the variable _year_targets_  
If needed, the selection of years can be changed here to include previous years.

In [ ]:
# choose the years to update
year_targets = years_available[-1:]  # Select last year!
log_msg(f"Aggregating '{target_table}' for years: {year_targets}")

In [ ]:
# Create the master table if it doesn't exist
with engine.begin() as conn:
    conn.execute(text(f'''
        CREATE TABLE IF NOT EXISTS public."{target_table}" (            
            "year" DOUBLE PRECISION,
            "PROVINCE" TEXT,  
            "ZONE" TEXT,
            "MALADIE" TEXT,
            "NUMSEM" INTEGER,            
            "Cas" INTEGER,            
            "Deces" INTEGER,   
            "ALERTE" DOUBLE PRECISION,
            "first_polygon" TEXT
        )
        PARTITION BY LIST ("year");
    '''))
print(f"Master table '{target_table}' ensured.")


In [ ]:
for year_target in year_targets:
        
    log_msg(f"Loading data from '{source_table}' for year {year_target}")
   
    # Partition name
    partition_name = f"{target_table}_{year_target}"

    # handle partition update
    if check_partition_exist(engine, target_table, year_target):
        with engine.begin() as conn:
            try:
                print(f"Partition for year {year_target} already exist, updating (truncate/insert) data..")
                conn.execute(text(f'TRUNCATE TABLE public."{partition_name}";'))
                conn.execute(text(f'INSERT INTO public."{partition_name}" SELECT * FROM public."{source_table}" WHERE "year" = {year_target};'))                
            except Exception as e:
                print(f"Error while updating the partition '{partition_name}'. {e}")
                raise 
    else:
        with engine.begin() as conn:
            try:
                print(f"Creating new partition '{partition_name}'..")
                conn.execute(text(f'CREATE TABLE public."{partition_name}" PARTITION OF public."{target_table}" FOR VALUES IN ({year_target});'))
                conn.execute(text(f'INSERT INTO public."{partition_name}" SELECT * FROM public."{source_table}" WHERE "year" = {year_target};'))                 
            except Exception as e:
                print(f"Error while creating the partition '{partition_name}'. {e}")
                raise 
    
    # clean    
    collected = gc.collect()
    print(collected)

In [ ]:
log_msg("Creating indexes..")
with engine.begin() as conn:
    for year in year_targets:
        table_name = f'{target_table}_{year}'
        index_name = f'idx_{target_table}_{year}_filters'
        print(index_name)
        sql = text(f'''CREATE INDEX IF NOT EXISTS "{index_name}" ON public."{table_name}" ("NUMSEM",  "PROVINCE", "ZONE", "MALADIE");''')
        conn.execute(sql)

In [ ]:
log_msg("Dropping view.")
with engine.begin() as conn:    
    sql = text(f'''DROP VIEW IF EXISTS public."view_dse_epi_agg_with_shapes";''')
    conn.execute(sql)

In [ ]:
log_msg(f"Update {target_table} finished!")